# Product Research Agent Evaluations with Arize Phoenix

This notebook demonstrates how to evaluate the Product Research Agent using Arize Phoenix tracing and evaluation capabilities.

## Overview
The Product Research Agent uses a 7-step LangGraph workflow to:
1. Parse and enhance user queries
2. Search for products via Tavily (Google Shopping)
3. Analyze product specifications and pricing
4. Analyze reviews and sentiment
5. Generate recommendations
6. Find alternative products
7. Synthesize final results

We'll evaluate:
- **Tool Calling**: Tavily shopping search accuracy
- **Workflow Convergence**: 7-step research completion
- **Recommendation Quality**: Product suggestion relevance
- **Performance**: Response time and cost metrics

## Setup and Imports

In [ ]:
import os
import sys
import asyncio
import time
from typing import List, Dict, Any

# Add src to path for imports
sys.path.append(os.path.join(os.getcwd(), 'src'))

# Phoenix imports
import phoenix as px
from phoenix.trace import SpanEvaluations
from phoenix.experiments import evaluate_experiment, run_experiment

# Our application imports
from src.agents.orchestrator import ProductResearchOrchestrator
from src.core.config import settings
from src.core.tracing import setup_phoenix_tracing, trace_product_research
from src.core.models import ResearchQuery, Product

# Standard libraries
import pandas as pd
import numpy as np
from datetime import datetime
import json

## Configuration and Phoenix Setup

In [ ]:
# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Check API keys
print("API Key Status:")
print(f"OpenAI: {'✅' if os.getenv('OPENAI_API_KEY') else '❌'}")
print(f"Tavily: {'✅' if os.getenv('TAVILY_API_KEY') else '❌'}")
print(f"Phoenix: {'✅' if os.getenv('PHOENIX_API_KEY') else '❌ (optional)'}")

In [ ]:
# Initialize Phoenix tracing
phoenix_success = setup_phoenix_tracing(
    api_key=os.getenv('PHOENIX_API_KEY'),
    space_id=os.getenv('PHOENIX_SPACE_ID'),
    project_name="product-research-agent-eval"
)

print(f"Phoenix Tracing: {'✅ Enabled' if phoenix_success else '❌ Disabled (using console)'}")

# Start Phoenix session (local UI)
session = px.launch_app()
print(f"Phoenix UI: {session.url}")

## Test Dataset: Product Research Queries

We'll create a diverse set of product research queries to evaluate our agent across different categories and complexity levels.

In [ ]:
# Define evaluation test cases
test_queries = [
    {
        "id": "laptop_programming",
        "query": "Best laptop under $2000 for programming and development work",
        "category": "computers",
        "expected_price_range": (700, 2000),
        "expected_features": ["programming", "development", "laptop"],
        "complexity": "medium"
    },
    {
        "id": "gaming_headset",
        "query": "Wireless gaming headset with good microphone under $150",
        "category": "gaming",
        "expected_price_range": (50, 150),
        "expected_features": ["wireless", "gaming", "headset", "microphone"],
        "complexity": "low"
    },
    {
        "id": "smartphone_camera",
        "query": "Smartphone with excellent camera for photography under $800",
        "category": "mobile",
        "expected_price_range": (300, 800),
        "expected_features": ["smartphone", "camera", "photography"],
        "complexity": "medium"
    },
    {
        "id": "home_office_chair",
        "query": "Ergonomic office chair for long work hours under $500",
        "category": "furniture",
        "expected_price_range": (100, 500),
        "expected_features": ["ergonomic", "office", "chair"],
        "complexity": "medium"
    },
    {
        "id": "kitchen_appliance",
        "query": "Best air fryer for family of 4 with easy cleanup",
        "category": "appliances",
        "expected_price_range": (50, 300),
        "expected_features": ["air fryer", "family", "cleanup"],
        "complexity": "low"
    }
]

print(f"Created {len(test_queries)} test queries for evaluation")
for query in test_queries:
    print(f"- {query['id']}: {query['query'][:50]}...")

## Initialize Product Research Agent

In [ ]:
# Initialize the orchestrator
orchestrator = ProductResearchOrchestrator()

print("Product Research Agent Status:")
print(f"LLM (GPT-5): {'✅' if orchestrator.llm else '❌'}")
print(f"Tavily Tool: {'✅' if orchestrator.tavily_tool.client else '❌'}")
print(f"Workflow: {'✅' if orchestrator.workflow else '❌'}")

## Evaluation Functions

We'll define evaluation functions to assess different aspects of our agent's performance.

In [ ]:
def evaluate_product_search_quality(result, expected_criteria):
    """
    Evaluate the quality of product search results.
    
    Returns a score from 0-1 based on:
    - Number of products found
    - Price range accuracy
    - Feature relevance
    """
    score = 0.0
    total_criteria = 4
    
    # 1. Products found (25%)
    if len(result.products) >= 3:
        score += 0.25
    elif len(result.products) >= 1:
        score += 0.15
    
    # 2. Price range accuracy (25%)
    if result.products:
        prices = [p.price for p in result.products if p.price]
        if prices:
            min_price, max_price = expected_criteria['expected_price_range']
            in_range_count = sum(1 for p in prices if min_price <= p <= max_price)
            if in_range_count > 0:
                score += 0.25 * (in_range_count / len(prices))
    
    # 3. Feature relevance (25%)
    if result.summary:
        summary_lower = result.summary.lower()
        matched_features = sum(1 for feature in expected_criteria['expected_features'] 
                             if feature.lower() in summary_lower)
        score += 0.25 * (matched_features / len(expected_criteria['expected_features']))
    
    # 4. Completion status (25%)
    if result.summary and result.recommendation:
        score += 0.25
    
    return min(score, 1.0)


def evaluate_response_time(research_time, complexity):
    """
    Evaluate response time based on query complexity.
    
    Target times:
    - Low complexity: < 10 seconds
    - Medium complexity: < 15 seconds  
    - High complexity: < 30 seconds
    """
    target_times = {
        "low": 10,
        "medium": 15,
        "high": 30
    }
    
    target = target_times.get(complexity, 15)
    
    if research_time <= target:
        return 1.0
    elif research_time <= target * 1.5:
        return 0.7
    elif research_time <= target * 2:
        return 0.4
    else:
        return 0.1


def evaluate_workflow_completion(result):
    """
    Evaluate if the 7-step workflow completed successfully.
    
    Checks for:
    - Products found
    - Reviews analyzed  
    - Recommendation generated
    - Alternatives provided
    - Summary created
    """
    score = 0.0
    total_steps = 5
    
    if result.products:
        score += 0.2
    
    if result.reviews:
        score += 0.2
        
    if result.recommendation:
        score += 0.2
        
    if result.alternatives:
        score += 0.2
        
    if result.summary:
        score += 0.2
        
    return score


print("Evaluation functions defined:")
print("- evaluate_product_search_quality")
print("- evaluate_response_time")
print("- evaluate_workflow_completion")

## Run Product Research Evaluations

Execute our test queries and collect evaluation metrics.

In [ ]:
async def run_product_research_evaluation():
    """
    Run the product research evaluation suite.
    """
    results = []
    
    print("Starting Product Research Agent Evaluation...\n")
    
    for i, test_case in enumerate(test_queries, 1):
        print(f"[{i}/{len(test_queries)}] Testing: {test_case['id']}")
        print(f"Query: {test_case['query']}")
        
        # Measure execution time
        start_time = time.time()
        
        try:
            # Run the research with Phoenix tracing
            with trace_product_research(test_case['query']) as span:
                result = await orchestrator.research_product(test_case['query'])
                
                # Add evaluation metadata to span
                if span:
                    span.set_attribute("eval.test_id", test_case['id'])
                    span.set_attribute("eval.category", test_case['category'])
                    span.set_attribute("eval.complexity", test_case['complexity'])
            
            execution_time = time.time() - start_time
            
            # Calculate evaluation scores
            search_quality = evaluate_product_search_quality(result, test_case)
            response_time_score = evaluate_response_time(execution_time, test_case['complexity'])
            workflow_completion = evaluate_workflow_completion(result)
            
            # Overall score (weighted average)
            overall_score = (
                search_quality * 0.5 + 
                response_time_score * 0.3 + 
                workflow_completion * 0.2
            )
            
            # Store results
            eval_result = {
                'test_id': test_case['id'],
                'query': test_case['query'],
                'category': test_case['category'],
                'complexity': test_case['complexity'],
                'execution_time': execution_time,
                'products_found': len(result.products),
                'search_quality_score': search_quality,
                'response_time_score': response_time_score,
                'workflow_completion_score': workflow_completion,
                'overall_score': overall_score,
                'has_recommendation': bool(result.recommendation),
                'has_alternatives': bool(result.alternatives),
                'research_result': result
            }
            
            results.append(eval_result)
            
            # Print summary
            print(f"✅ Completed in {execution_time:.2f}s")
            print(f"   Products: {len(result.products)} | Overall Score: {overall_score:.2f}")
            print(f"   Quality: {search_quality:.2f} | Speed: {response_time_score:.2f} | Completion: {workflow_completion:.2f}\n")
            
        except Exception as e:
            print(f"❌ Failed: {str(e)}\n")
            
            # Store failure result
            eval_result = {
                'test_id': test_case['id'],
                'query': test_case['query'],
                'category': test_case['category'],
                'complexity': test_case['complexity'],
                'execution_time': time.time() - start_time,
                'products_found': 0,
                'search_quality_score': 0.0,
                'response_time_score': 0.0,
                'workflow_completion_score': 0.0,
                'overall_score': 0.0,
                'has_recommendation': False,
                'has_alternatives': False,
                'error': str(e),
                'research_result': None
            }
            results.append(eval_result)
    
    return results

# Run the evaluation
evaluation_results = await run_product_research_evaluation()

## Evaluation Results Analysis

In [ ]:
# Convert results to DataFrame for analysis
df_results = pd.DataFrame([{
    'test_id': r['test_id'],
    'category': r['category'],
    'complexity': r['complexity'],
    'execution_time': r['execution_time'],
    'products_found': r['products_found'],
    'search_quality': r['search_quality_score'],
    'response_time': r['response_time_score'],
    'workflow_completion': r['workflow_completion_score'],
    'overall_score': r['overall_score'],
    'has_recommendation': r['has_recommendation'],
    'has_alternatives': r['has_alternatives']
} for r in evaluation_results])

print("📊 EVALUATION RESULTS SUMMARY")
print("=" * 50)

# Overall metrics
print(f"Total Tests: {len(df_results)}")
print(f"Success Rate: {(df_results['overall_score'] > 0).sum()}/{len(df_results)} ({((df_results['overall_score'] > 0).sum()/len(df_results)*100):.1f}%)")
print(f"Average Overall Score: {df_results['overall_score'].mean():.3f}")
print(f"Average Response Time: {df_results['execution_time'].mean():.2f}s")
print(f"Average Products Found: {df_results['products_found'].mean():.1f}")

print("\n📈 SCORE BREAKDOWN")
print("-" * 30)
print(f"Search Quality: {df_results['search_quality'].mean():.3f}")
print(f"Response Time: {df_results['response_time'].mean():.3f}")
print(f"Workflow Completion: {df_results['workflow_completion'].mean():.3f}")

print("\n🎯 BY CATEGORY")
print("-" * 20)
category_stats = df_results.groupby('category').agg({
    'overall_score': 'mean',
    'execution_time': 'mean',
    'products_found': 'mean'
}).round(3)
print(category_stats)

print("\n⚡ BY COMPLEXITY")
print("-" * 20)
complexity_stats = df_results.groupby('complexity').agg({
    'overall_score': 'mean',
    'execution_time': 'mean',
    'products_found': 'mean'
}).round(3)
print(complexity_stats)

## Detailed Results Table

In [ ]:
# Display detailed results
display_df = df_results[[
    'test_id', 'category', 'complexity', 'execution_time', 
    'products_found', 'overall_score', 'search_quality',
    'response_time', 'workflow_completion'
]].round(3)

print("📋 DETAILED EVALUATION RESULTS")
print("=" * 80)
print(display_df.to_string(index=False))

## Sample Research Results

Let's examine a successful research result in detail.

In [ ]:
# Find the best performing result
best_result = max(evaluation_results, key=lambda x: x['overall_score'])

if best_result['research_result']:
    result = best_result['research_result']
    
    print(f"🏆 BEST PERFORMING RESULT: {best_result['test_id']}")
    print(f"Query: {best_result['query']}")
    print(f"Overall Score: {best_result['overall_score']:.3f}")
    print("=" * 60)
    
    print(f"\n📊 METRICS:")
    print(f"Products Found: {len(result.products)}")
    print(f"Execution Time: {best_result['execution_time']:.2f}s")
    print(f"Has Recommendation: {bool(result.recommendation)}")
    print(f"Has Alternatives: {len(result.alternatives)} alternatives")
    
    if result.products:
        print(f"\n🛍️ SAMPLE PRODUCTS:")
        for i, product in enumerate(result.products[:3], 1):
            print(f"{i}. {product.name}")
            print(f"   Price: ${product.price or 'N/A'}")
            print(f"   Brand: {product.brand or 'Unknown'}")
            print(f"   URL: {product.url[:60] if product.url else 'N/A'}...")
    
    if result.recommendation:
        print(f"\n💡 RECOMMENDATION:")
        print(result.recommendation[:300] + "..." if len(result.recommendation) > 300 else result.recommendation)
    
    if result.summary:
        print(f"\n📝 SUMMARY:")
        print(result.summary[:400] + "..." if len(result.summary) > 400 else result.summary)
else:
    print("No successful results to display.")

## Phoenix Trace Analysis

If Phoenix tracing is enabled, you can view detailed traces in the Phoenix UI.

In [ ]:
print("🔍 PHOENIX TRACE ANALYSIS")
print("=" * 40)

if phoenix_success:
    print(f"Phoenix UI: {session.url}")
    print("\nIn the Phoenix UI, you can:")
    print("• View detailed traces for each research query")
    print("• Analyze the 7-step workflow execution")
    print("• Monitor tool calling performance (Tavily API)")
    print("• Track token usage and costs")
    print("• Compare performance across test cases")
    
    print("\n📈 Key Metrics to Monitor:")
    print("• LLM call latency and token usage")
    print("• Tavily API response times")
    print("• Workflow step completion rates")
    print("• Error rates and failure modes")
else:
    print("Phoenix tracing not enabled - check console output for basic traces")
    print("To enable Phoenix tracing:")
    print("1. Set PHOENIX_API_KEY in your .env file")
    print("2. Set PHOENIX_SPACE_ID in your .env file")
    print("3. Restart the notebook")

## Evaluation Insights and Recommendations

Based on the evaluation results, here are key insights and recommendations for improving the Product Research Agent.

In [ ]:
# Generate insights based on results
def generate_evaluation_insights(results_df):
    insights = []
    
    # Overall performance
    avg_score = results_df['overall_score'].mean()
    if avg_score >= 0.8:
        insights.append("✅ Excellent overall performance (>80% average score)")
    elif avg_score >= 0.6:
        insights.append("⚠️ Good performance with room for improvement (60-80% average score)")
    else:
        insights.append("❌ Performance needs significant improvement (<60% average score)")
    
    # Response time analysis
    avg_time = results_df['execution_time'].mean()
    if avg_time <= 10:
        insights.append("⚡ Excellent response times (<10s average)")
    elif avg_time <= 20:
        insights.append("🕒 Good response times (10-20s average)")
    else:
        insights.append("🐌 Slow response times (>20s average) - consider optimization")
    
    # Product discovery
    avg_products = results_df['products_found'].mean()
    if avg_products >= 5:
        insights.append("🛍️ Excellent product discovery (5+ products per query)")
    elif avg_products >= 3:
        insights.append("🔍 Good product discovery (3-5 products per query)")
    else:
        insights.append("📦 Limited product discovery (<3 products per query)")
    
    # Workflow completion
    avg_completion = results_df['workflow_completion'].mean()
    if avg_completion >= 0.9:
        insights.append("✅ Excellent workflow completion rate (>90%)")
    elif avg_completion >= 0.7:
        insights.append("⚠️ Good workflow completion rate (70-90%)")
    else:
        insights.append("❌ Poor workflow completion rate (<70%)")
    
    return insights

# Generate recommendations
def generate_recommendations(results_df):
    recommendations = []
    
    # Based on response time
    if results_df['response_time'].mean() < 0.7:
        recommendations.append("Consider optimizing API calls and reducing LLM token usage")
    
    # Based on search quality
    if results_df['search_quality'].mean() < 0.7:
        recommendations.append("Improve product extraction and price parsing algorithms")
        recommendations.append("Enhance query preprocessing for better search results")
    
    # Based on workflow completion
    if results_df['workflow_completion'].mean() < 0.8:
        recommendations.append("Add better error handling and recovery mechanisms")
        recommendations.append("Implement fallback strategies for failed API calls")
    
    # Category-specific recommendations
    category_performance = results_df.groupby('category')['overall_score'].mean()
    worst_category = category_performance.idxmin()
    if category_performance[worst_category] < 0.6:
        recommendations.append(f"Focus on improving {worst_category} category performance")
    
    return recommendations

# Display insights
insights = generate_evaluation_insights(df_results)
recommendations = generate_recommendations(df_results)

print("💡 EVALUATION INSIGHTS")
print("=" * 30)
for insight in insights:
    print(f"• {insight}")

print("\n🎯 RECOMMENDATIONS")
print("=" * 25)
if recommendations:
    for rec in recommendations:
        print(f"• {rec}")
else:
    print("• Performance looks good! Consider expanding test coverage.")

print("\n🚀 NEXT STEPS")
print("=" * 15)
print("• Monitor Phoenix traces for deeper performance analysis")
print("• Expand test dataset with more diverse queries")
print("• Implement A/B testing for different prompt strategies")
print("• Set up automated evaluation pipeline for continuous monitoring")
print("• Create custom evaluation metrics for domain-specific requirements")

## Export Results for Further Analysis

In [ ]:
# Save results to files
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save summary metrics
df_results.to_csv(f'evaluation_results_{timestamp}.csv', index=False)
print(f"✅ Saved evaluation results to: evaluation_results_{timestamp}.csv")

# Save detailed results with full research data
detailed_results = {
    'timestamp': timestamp,
    'summary_metrics': {
        'total_tests': len(df_results),
        'success_rate': (df_results['overall_score'] > 0).sum() / len(df_results),
        'average_score': df_results['overall_score'].mean(),
        'average_response_time': df_results['execution_time'].mean(),
        'average_products_found': df_results['products_found'].mean()
    },
    'test_results': evaluation_results,
    'insights': insights,
    'recommendations': recommendations
}

with open(f'detailed_evaluation_{timestamp}.json', 'w') as f:
    # Convert any non-serializable objects to strings
    serializable_results = detailed_results.copy()
    for result in serializable_results['test_results']:
        if 'research_result' in result and result['research_result']:
            # Convert Pydantic models to dict
            result['research_result'] = result['research_result'].dict() if hasattr(result['research_result'], 'dict') else str(result['research_result'])
    
    json.dump(serializable_results, f, indent=2, default=str)

print(f"✅ Saved detailed results to: detailed_evaluation_{timestamp}.json")

print("\n📊 EVALUATION COMPLETE")
print("=" * 30)
print(f"Timestamp: {timestamp}")
print(f"Total Tests: {len(df_results)}")
print(f"Average Score: {df_results['overall_score'].mean():.3f}")
print(f"Phoenix UI: {session.url if phoenix_success else 'Not available'}")
print("\nResults saved for further analysis and monitoring.")